# **Ottimizzazione di Performance Cognitive o Sportive tramite Machine Learning su Dati Biometrici**

<div style="border-left: 3px solid #000; padding: 5px 15px; margin: 10px 0;">
<span style="color: #666; font-size: 19px; letter-spacing: 1.5px;">PROGETTO INGEGNERIA INFORMATICA · A.A. 2025/2026</span>
</div>

**Luis Mariduena** & **Edoardo Sorgentone**

---

# **Explainability Pipeline**

## **1. Ricevuta dati dei modelli**

Apriamo i dataframe e modelli generati dalla Model Development pipeline

In [2]:
import joblib
from google.colab import drive

drive.mount('/content/drive')


datiAllenamento = joblib.load("/content/drive/MyDrive/PII_folder/export_Bundle.pkl")

modelli = datiAllenamento["results"]
X_scaled = datiAllenamento["X_scaled"]
y_targets = datiAllenamento["targets"]


Mounted at /content/drive


## **2. Scelta feature importanti e generazione grafici importanti con SHAP**

Come prima fase della generazione di dati per la explainability. Abbiamo deciso di applicare SHAP per analizzare il contributo delle feature identificate come rilevanti dal metodo di feature selection dei modelli piu performanti nella sezione di allenamento.

In [ ]:
!pip install shap
import os
import shap
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np


basepath = "/content/drive/MyDrive/PII_folder/Risultati_SHAP"
risultatiSHAP = {}

#Usa l'explainer shap per tutti I modelli
for areaCog,grupoModelli in modelli.items():

  risultatiSHAP[areaCog] = {}

  for priorita,modelloDati in grupoModelli.items():
#Passo dati del dict a variabili
    modello = modelloDati["model_object"]
    features = modelloDati["selected_features"]
    X_subset = X_scaled[features]
#Calcolo valori SHAP
    explainer = shap.Explainer(modello.predict, X_subset)
    shap_values = explainer(X_subset)

#sceglie le feature piu importanti e li salva importante per ALE e Dice dopo
    importance_df = pd.DataFrame({
        'feature': X_subset.columns,
        'importance': np.abs(shap_values.values).mean(axis=0)
    })
#elimina features che hanno influenza nulla
    importance_df = importance_df[importance_df["importance"]>0]

    importance_df = importance_df.sort_values(by='importance', ascending=False)

    featRel = importance_df.head(7)["feature"].tolist()
#Salva I risultati SHAP a un dict in caso di dover continuare in altra sessione
    risultatiSHAP[areaCog][priorita] = {
        "featRelevanti":featRel,
        "explainer": explainer,
        "risultatiShap": shap_values
    }
#Crea folder dove si salveranno tutti i dati
    folderName = f"{areaCog}_{priorita}"
    folderPath = os.path.join(basepath, folderName)
    os.makedirs(folderPath, exist_ok=True)

#Generazione Grafici
    plt.figure(figsize=(18, 10))
    shap.plots.beeswarm(shap_values, show=False)
    plt.gcf().subplots_adjust(left=0.35)
    plt.savefig(os.path.join(folderPath, "beeswarm_plot.png"),bbox_inches='tight', dpi=300)
    plt.close()

    plt.figure(figsize=(18, 10))
    shap.plots.bar(shap_values, show=False)
    plt.gcf().subplots_adjust(left=0.35)
    plt.savefig(os.path.join(folderPath, "bar_plot.png"),bbox_inches='tight', dpi=300)
    plt.close()

    plt.figure(figsize=(18, 10))
    shap.plots.waterfall(shap_values[0], show=False)
    plt.gcf().subplots_adjust(left=0.35)
    plt.savefig(os.path.join(folderPath,"waterfall_plot.png"),bbox_inches='tight', dpi=300)
    plt.close()

#salva dict con i Risultati dei explainer in caso sia utili
joblib.dump(risultatiSHAP, "/content/drive/MyDrive/PII_folder/Risultati_SHAP/risultatiSHAP.pkl")

['/content/drive/MyDrive/PII_folder/Risultati_SHAP/risultatiSHAP.pkl']

## **3. Generazione grafici con ALE**

Dopo aver usato SHAP per determinare le feature piu relevanti salvarle al array featREl, e generare grafici relevanti per l'analisi futuro proseguiamo con la generazione di grafici su queste feature relevanti usando ALE. ALE dovrebbe permetterci di capire come la variazione delle feature a cambi grandi influenza il risultato dello score dei modelli e se c'e' un punto dove un incremento di un feature che aveva un effetto positivo diventa negativo e il viceversa.

In [ ]:
!pip install PyALE
from PyALE import ale

basepath = "/content/drive/MyDrive/PII_folder/Risultati_ALE"


for areaCog,grupoModelli in modelli.items():

  for priorita,modelloDati in grupoModelli.items():

    folder_name = f"{areaCog}_{priorita}"

    folderPath = os.path.join(
            basepath,
            folder_name
    )

    os.makedirs(folderPath, exist_ok=True)

    modello = modelloDati["model_object"]
    features = modelloDati["selected_features"]
    X_subset = X_scaled[features]
    featIMP = risultatiSHAP[areaCog][priorita]["featRelevanti"]

#generazione grafici ALE
    for feat in featIMP:

      fig, ax = plt.subplots(figsize=(10,6))

      ale(
          X=X_subset,
          model=modello,
          feature=[feat],
          grid_size=20,
          include_CI=False,
          plot=True,
          fig=fig,
          ax=ax
      )
      plt.title(f"ALE - {feat}")
      plt.tight_layout()
      plt.savefig(
          os.path.join(folderPath,f"ale_{feat}.png"),
          bbox_inches="tight",
          dpi=300
      )

      plt.close()




INFO:PyALE._ALE_generic:Continuous feature detected.
INFO:PyALE._ALE_generic:Continuous feature detected.
INFO:PyALE._ALE_generic:Continuous feature detected.
INFO:PyALE._ALE_generic:Continuous feature detected.
INFO:PyALE._ALE_generic:Continuous feature detected.
INFO:PyALE._ALE_generic:Continuous feature detected.
INFO:PyALE._ALE_generic:Continuous feature detected.
INFO:PyALE._ALE_generic:Continuous feature detected.
INFO:PyALE._ALE_generic:Continuous feature detected.
INFO:PyALE._ALE_generic:Continuous feature detected.
INFO:PyALE._ALE_generic:Continuous feature detected.
INFO:PyALE._ALE_generic:Continuous feature detected.
INFO:PyALE._ALE_generic:Continuous feature detected.
INFO:PyALE._ALE_generic:Continuous feature detected.
INFO:PyALE._ALE_generic:Continuous feature detected.
INFO:PyALE._ALE_generic:Continuous feature detected.
INFO:PyALE._ALE_generic:Continuous feature detected.
INFO:PyALE._ALE_generic:Continuous feature detected.
INFO:PyALE._ALE_generic:Continuous feature det

## **4. Generazione dati con DiCe**

Mentre ALE ci permette rispondere la domanda su quale effetto hanno le feature ad un livello globale non ci dice la lore influenza ad un livello locale. Per rispondere questa seconda domando abbiamo deciso di usare DiCe sulle 3 feature determinate come piu importanti da SHAP.

### Generazione Counter Factuals

In [ ]:
!pip install dice-ml

import dice_ml
from dice_ml import Dice

basepath = "/content/drive/MyDrive/PII_folder/Risultati_DiCe"


for areaCog,grupoModelli in modelli.items():

  for priorita,modelloDati in grupoModelli.items():

    modello = modelloDati["model_object"]
    features = modelloDati["selected_features"]
    X_subset = X_scaled[features].copy()
    #scegliamo i 3 primi features per fare DiCe
    feat_to_vary = risultatiSHAP[areaCog][priorita]["featRelevanti"][:5]

    if hasattr(modello, "predict_proba"):
      #crea la cartella per i counterfactuals
      folder_name = f"{areaCog}_{priorita}"

      folderPath = os.path.join(
            basepath,
            folder_name
      )
      os.makedirs(folderPath, exist_ok=True)
      #creazione inputs per dice
      X_subset = X_subset.reset_index(drop=True)
      target = y_targets[areaCog[:3]].reset_index(drop=True)
      dice_df = X_subset.copy()
      dice_df["target"] = target.values
      data_dice = dice_ml.Data(dataframe=dice_df, continuous_features=features, outcome_name="target")
      model_dice = dice_ml.Model(model=modello, backend= "sklearn")
      exp = Dice(data_dice, model_dice, method="random")

      #randomizer per instances del counterfactual
      All_Results = []

      sampled_instances = X_subset.sample(7)
      cols_to_keep = feat_to_vary + ["target"]

      for idx in sampled_instances.index:
        failed_counterfactuals = []

        query_instance = X_subset.iloc[[idx]]
        og_df = query_instance.copy()
        og_df["type"] = "originale"
        og_df["grupo"] = idx
        og_df["target"] = target.loc[idx]
        og_df = og_df[cols_to_keep]
        #generazione counterfactuals
        for i in range(3):
          if i == target.loc[idx]:
            continue
        #se trova counter factual continua altrimenti lo salva in un array
          try:
            dice_exp = exp.generate_counterfactuals(query_instance, total_CFs=5, desired_class=i, features_to_vary=feat_to_vary)

          except Exception as e:
            failed_counterfactuals.append({
                "index":idx,
                "targetClass":i
                })
            print(f"Errore durante la generazione dei counterfactuals per l'indice {idx}: {e}")
            continue

          #gen df di cf temporale
          cf_df = dice_exp.cf_examples_list[0].final_cfs_df
          cf_df = cf_df[cols_to_keep]
          cf_df["type"] = "counterfactuale"
          cf_df["grupo"] = idx

          #gen info feat modificate
          changed_features = []

          for j in range (len(cf_df)):

            changes = []

            for feat in feat_to_vary:
              original_var = query_instance.iloc[0][feat]

              cf_val = cf_df.iloc[j][feat]
              if original_var != cf_val:
                delta = round(cf_val - original_var, 3)

                changes.append(f"{feat}:{delta:+}")
            changed_features.append("|".join(changes))

          cf_df["feat_modifichate"] = changed_features

          #aggiungere a df finale
          combined = pd.concat([og_df, cf_df], ignore_index=True)

          All_Results.append(combined)

          spacer = pd.DataFrame(
            [[None] * len(combined.columns)],
            columns=combined.columns
          )
          All_Results.append(spacer)
      failed_df = pd.DataFrame(failed_counterfactuals)

      failed_df.to_csv(
        os.path.join(folderPath, "failed_indices.csv"),
        index=False
      )

      final_df = pd.concat(
        All_Results,
        ignore_index=True
      )

      final_df.to_csv(
       os.path.join(folderPath, "all_counterfactuals.csv"),
       index=False
      )

  0%|          | 0/1 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/dice_ml/explainer_interfaces/dice_random.py:165: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_instance_df[self.data_interface.outcome_name] = \
100%|██████████| 1/1 [00:00<00:00,  3.66it/s]
/tmp/ipykernel_478/438570634.py:71: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cf_df["type"] = "counterfactuale"
/tmp/ipykernel_478/438570634.py:72: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try us

No Counterfactuals found for the given configuration, perhaps try with different parameters... ; total time taken: 00 min 00 sec
Errore durante la generazione dei counterfactuals per l'indice 82: No counterfactuals found for any of the query points! Kindly check your configuration.


  0%|          | 0/1 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/dice_ml/explainer_interfaces/dice_random.py:165: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_instance_df[self.data_interface.outcome_name] = \
100%|██████████| 1/1 [00:00<00:00,  2.70it/s]
/tmp/ipykernel_478/438570634.py:71: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cf_df["type"] = "counterfactuale"
/tmp/ipykernel_478/438570634.py:72: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try us

No Counterfactuals found for the given configuration, perhaps try with different parameters... ; total time taken: 00 min 00 sec
Errore durante la generazione dei counterfactuals per l'indice 68: No counterfactuals found for any of the query points! Kindly check your configuration.


  0%|          | 0/1 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/dice_ml/explainer_interfaces/dice_random.py:165: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_instance_df[self.data_interface.outcome_name] = \
100%|██████████| 1/1 [00:00<00:00,  3.44it/s]
/tmp/ipykernel_478/438570634.py:71: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cf_df["type"] = "counterfactuale"
/tmp/ipykernel_478/438570634.py:72: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try us

No Counterfactuals found for the given configuration, perhaps try with different parameters... ; total time taken: 00 min 00 sec
Errore durante la generazione dei counterfactuals per l'indice 37: No counterfactuals found for any of the query points! Kindly check your configuration.


  0%|          | 0/1 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/dice_ml/explainer_interfaces/dice_random.py:165: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_instance_df[self.data_interface.outcome_name] = \
100%|██████████| 1/1 [00:00<00:00,  3.10it/s]
/tmp/ipykernel_478/438570634.py:71: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cf_df["type"] = "counterfactuale"
/tmp/ipykernel_478/438570634.py:72: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try us

No Counterfactuals found for the given configuration, perhaps try with different parameters... ; total time taken: 00 min 00 sec
Errore durante la generazione dei counterfactuals per l'indice 2: No counterfactuals found for any of the query points! Kindly check your configuration.


  0%|          | 0/1 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/dice_ml/explainer_interfaces/dice_random.py:165: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_instance_df[self.data_interface.outcome_name] = \
100%|██████████| 1/1 [00:00<00:00,  2.06it/s]
/tmp/ipykernel_478/438570634.py:71: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cf_df["type"] = "counterfactuale"
/tmp/ipykernel_478/438570634.py:72: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try us

### Data Processing di counterfactuals generati

Codice per generazione Tabelle e Grafici per interpretare risultati di DiCe

In [9]:
import pandas as pd
from collections import Counter
from collections import defaultdict
import os
import matplotlib.pyplot as plt

#path dei counterfactuals
path_rea = "/content/drive/MyDrive/PII_folder/Risultati_DiCe/reasoning_secondary/all_counterfactuals.csv"
path_ver = "/content/drive/MyDrive/PII_folder/Risultati_DiCe/verbal_secondary/all_counterfactuals.csv"

#apertura files
df_rea = pd.read_csv(path_rea)
df_ver = pd.read_csv(path_ver)

#dictionary per automatizzare il processo
df_list = {
    "/content/drive/MyDrive/PII_folder/Risultati_DiCe/reasoning_secondary":df_rea,
    "/content/drive/MyDrive/PII_folder/Risultati_DiCe/verbal_secondary":df_ver
    }

for path,df in df_list.items():
  #Feature Modification Frequency tabella e grafico

  feature_counter = Counter()

  for changes in df["feat_modifichate"].dropna():
      for item in str(changes).split("|"):
          if ":" in item:
              feature = item.split(":")[0]
              feature_counter[feature] += 1

  frequency_df = (
      pd.DataFrame(
          feature_counter.items(),
          columns=["Feature", "Modification Frequency"]
      )
      .sort_values("Modification Frequency", ascending=False)
      .reset_index(drop=True)
  )
  #Generazione Grafico
  plt.figure(figsize=(10, 6))
  plt.barh(
    frequency_df["Feature"],
    frequency_df["Modification Frequency"]
  )
  plt.xlabel("Modification Frequency")
  plt.ylabel("Feature")
  plt.title("Feature Modification Frequency")
  plt.tight_layout()
  plt.savefig(os.path.join(path, "feature_modification_frequency.png"),
    dpi=300,
    bbox_inches="tight"
  )
  plt.close()
  #Salvamento tabella
  frequency_df.to_csv(
      os.path.join(path, "modifications_frequency.csv"),
      index=False
  )

  #Mean Required Change Tabella e Grafico
  feature_changes = defaultdict(list)

  for changes in df["feat_modifichate"].dropna():
      for item in str(changes).split("|"):
          if ":" in item:
              feature, value = item.split(":")
              feature_changes[feature].append(abs(float(value)))

  mean_change_df = pd.DataFrame({
      "Feature": feature_changes.keys(),
      "Mean Required Change": [
          sum(v)/len(v) for v in feature_changes.values()
      ]
  })

  mean_change_df = (
      mean_change_df
      .sort_values("Mean Required Change", ascending=False)
      .reset_index(drop=True)
  )
  #Generazione Grafico
  plt.figure(figsize=(10, 6))
  plt.barh(
    mean_change_df["Feature"],
    mean_change_df["Mean Required Change"]
  )
  plt.xlabel("Mean Required Change")
  plt.ylabel("Feature")
  plt.title("Mean Required Change by Feature")
  plt.tight_layout()
  plt.savefig(
    os.path.join (path,"mean_required_change.png"),
    dpi=300,
    bbox_inches="tight"
  )
  plt.close()
  #Salvamento Tabella
  mean_change_df.to_csv(
      os.path.join(path, "mean_required_change.csv"),
      index=False
  )